# Chapter 07 — Unit Testing, Docker Flex Templates, CI/CD & Security

Welcome to the interactive notebook for **Chapter 07**!

> 📖 **Concept Reference**: For deep dives into `TestStream` watermark testing, Docker Flex Templates, Cloud Build CI/CD, and Private IP/CMEK security hardening, see [Chapter 07 README](README.md).

### Interactive Workflow
1. Run deterministic pipeline unit tests with `TestPipeline` and `PAssert`.
2. Test unbounded event-time windowing and watermarks programmatically with `TestStream`.
3. Inspect the containerized Flex Template specification and parameters.
4. Verify enterprise security hardening (Private IPs and CMEK encryption).

## 1. Environment Setup & Kernel Selection

This project uses **[`uv`](https://docs.astral.sh/uv/)** to manage dependencies in an isolated `.venv` virtual environment.

> 📘 **Full Environment Setup Guide**: For complete setup instructions, terminal commands, and kernel registration, see the root [README.md](../README.md) (*Section: Environment Setup & Activating the Kernel with uv*).

* **Option A: In the Terminal**: Activate with `source .venv/bin/activate` or execute commands directly with `uv run`.
* **Option B: In This Jupyter Notebook (Kernel Selection)**:
  If you encounter `ModuleNotFoundError: No module named 'apache_beam'`, register the kernel from your terminal:
  ```bash
  uv run python -m ipykernel install --user --name dataflow-workshops --display-name "Python (dataflow-workshops)"
  ```
  Then click the **Kernel Picker** in the top-right corner of this notebook editor and select **`Python (dataflow-workshops)`** (or browse to `.venv/bin/python`).

Run the verification cell below to confirm your setup:

In [1]:
import os
import sys

print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is running inside the project's .venv!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Select 'Python (dataflow-workshops)' from the top-right kernel picker.")

import apache_beam as beam
print(f"✅ Apache Beam Version: {beam.__version__}")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is running inside the project's .venv!


✅ Apache Beam Version: 2.77.0


## 2. Architecture & Visual DAG

Here is the architecture for this chapter, illustrating unit testing, Cloud Build CI/CD, Artifact Registry packaging, and enterprise security boundaries:

![Chapter 07 Testing, Flex Templates & CI/CD DAG](../artifacts/07_testing_templates_and_cicd_dag.png)

<details>
<summary><b>Click to expand Mermaid source code</b></summary>

```mermaid
flowchart LR
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;
    classDef sink fill:#e6f4ea,stroke:#34a853,stroke-width:2px,color:#137333,font-weight:bold;
    classDef test fill:#f3e8fd,stroke:#9334e6,stroke-width:2px,color:#6b21a8,font-weight:bold;
    classDef security fill:#fee2e2,stroke:#ef4444,stroke-width:2px,color:#991b1b,font-weight:bold;

    subgraph Testing["1. Deterministic Testing"]
        STREAM["TestStream<br/>Watermark advances"]:::test
        PIPE["TestPipeline<br/>In-memory execution"]:::test
        ASSERT["PAssert / assert_that"]:::test
        STREAM --> PIPE --> ASSERT
    end

    subgraph CICD["2. Automated CI/CD & Flex Templates"]
        GIT[("Git Repository")]:::pcoll
        CB_TEST["Cloud Build: Pytest"]:::ptrans
        CB_DOCKER["Docker Build & Push"]:::ptrans
        AR[("Artifact Registry Image")]:::pcoll
        GCS[("GCS Template Spec")]:::sink
        GIT --> CB_TEST --> CB_DOCKER --> AR --> GCS
    end

    subgraph Security["3. Enterprise Security Boundary"]
        VPC["Private VPC (No Public IPs)"]:::security
        WORKERS["Dataflow Workers (--no-use-public-ips)"]:::security
        CMEK["Cloud KMS (CMEK Encryption)"]:::security
        BQ_SECURE[("BigQuery CMEK Encrypted")]:::sink
        VPC --> WORKERS
        CMEK --> WORKERS --> BQ_SECURE
    end
```
</details>

## 3. Unit Testing with `TestPipeline` and `PAssert`

We test transform business logic deterministically in memory.
*(For assertions and test suite structure, see [Chapter 07 README](README.md)).*

In [2]:
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.util import assert_that, equal_to

class MultiplyByTwoFn(beam.DoFn):
    def process(self, element):
        yield element * 2

with TestPipeline() as p:
    numbers = p | beam.Create([1, 2, 3, 4, 5])
    doubled = numbers | beam.ParDo(MultiplyByTwoFn())
    
    # Deterministic assertion on distributed PCollection
    assert_that(doubled, equal_to([2, 4, 6, 8, 10]))

print("✅ TestPipeline assertion passed!")

✅ TestPipeline assertion passed!


## 4. Unbounded Stream Testing with `TestStream`

We programmatically advance watermarks and test window closures without GCP dependencies.
*(For TestStream mechanics and trigger assertions, see [Chapter 07 README](README.md)).*

In [3]:
from apache_beam.testing.test_stream import TestStream
from apache_beam.transforms.window import FixedWindows, TimestampedValue

# Define event stream: 2 elements in window 1, then watermark advances to 60s, then 1 element in window 2
stream = (
    TestStream()
    .add_elements([
        TimestampedValue(("bob", 10.0), timestamp=10),
        TimestampedValue(("bob", 20.0), timestamp=25),
    ])
    .advance_watermark_to(60)
    .add_elements([
        TimestampedValue(("bob", 30.0), timestamp=75),
    ])
    .advance_watermark_to_infinity()
)

with TestPipeline() as p:
    totals = (
        p
        | stream
        | beam.WindowInto(FixedWindows(60))
        | beam.CombinePerKey(sum)
    )
    
    # Window 1 emits ('bob', 30.0), Window 2 emits ('bob', 30.0)
    assert_that(totals, equal_to([("bob", 30.0), ("bob", 30.0)]))

print("✅ TestStream streaming window assertion passed!")

✅ TestStream streaming window assertion passed!


## 5. Containerized Dataflow Flex Templates

Package Python pipelines into portable Docker containers stored on Artifact Registry.
*(For template metadata specification and launch parameters, see [Chapter 07 README](README.md)).*

In [4]:
from pathlib import Path
import json

print("=== Flex Template Dockerfile ===")
print(Path("template/Dockerfile").read_text())

print("=== Flex Template Metadata Parameters ===")
metadata = json.loads(Path("template/metadata.json").read_text())
for param in metadata["parameters"]:
    print(f"Parameter: {param['name']:<20} | Optional: {param['isOptional']} | {param['helpText']}")

=== Flex Template Dockerfile ===
# ==============================================================================
# Dataflow Flex Template Dockerfile — Chapter 07
# Base image contains the Dataflow Flex Template launcher harness
# ==============================================================================
FROM gcr.io/dataflow-templates-base/python311-template-launcher-base:latest

ARG WORKDIR=/template
RUN mkdir -p ${WORKDIR}
WORKDIR ${WORKDIR}

# Install dependencies
COPY requirements.txt ${WORKDIR}/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt

# Copy pipeline code
COPY pipelines/ ${WORKDIR}/pipelines/

# Configure the entrypoint script location for the Dataflow Flex Template harness
ENV FLEX_TEMPLATE_PYTHON_PY_FILE="${WORKDIR}/pipelines/secure_orders_pipeline.py"
ENV FLEX_TEMPLATE_PYTHON_REQUIREMENTS_FILE="${WORKDIR}/requirements.txt"

ENTRYPOINT ["/opt/google/dataflow/python_template_launcher"]

=== Flex Template Metadata Parameters ===
Parameter: input_sub

## 6. Enterprise Security Hardening (Private IPs & CMEK)

Configure network isolation with `--no_use_public_ips` and encryption with `--dataflow_kms_key`.
*(For VPC configuration and Cloud KMS key rings, see [Chapter 07 README](README.md)).*

## 7. Running the Full Unit Test Suite with Pytest

Let's execute the complete unit test suite via `pytest` to verify both transform unit tests and streaming `TestStream` tests:

In [5]:
import subprocess

cmd = ["bash", "scripts/01_run_unit_tests.sh"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

 Chapter 07: Running Unit Tests (TestPipeline, PAssert & TestStream)
============================= test session starts ==============================
platform darwin -- Python 3.12.13, pytest-9.1.1, pluggy-1.7.0 -- /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/zelda.luconi/Medium/google-products/dataflow-workshops
configfile: pyproject.toml
plugins: platformdirs-4.13.6, anyio-4.15.1
collecting ... collected 3 items

tests/test_streaming_teststream.py::TestStreamingPipelines::test_windowing_with_test_stream PASSED [ 33%]
tests/test_transforms.py::TestOrderTransforms::test_tax_calculation_negative_subtotal_dlq PASSED [ 66%]
tests/test_transforms.py::TestOrderTransforms::test_tax_calculation_success PASSED [100%]

=============================== warnings summary ===============================
../.venv/lib/python3.12/site-packages/apache_beam/testing/test_pipeline.py:37
  /Users/zelda.luconi/Medium/google-products/da

## 8. Summary: The Complete Dataflow Engineering Mastery

Congratulations! You have completed all 7 chapters of the **Google Cloud Dataflow Workshop**:

| Chapter | Key Competencies Mastered |
| :--- | :--- |
| **Chapter 01: Beam Fundamentals** | Pipelines, PCollections, PTransforms, DirectRunner vs. DataflowRunner. |
| **Chapter 02: Side Inputs & BigQuery** | `DoFn` lifecycle (`setup`/`teardown`), `beam.pvalue.AsDict`, DLQ Tagged Outputs, BigQuery Storage Write API. |
| **Chapter 03: Streaming & Windowing** | Pub/Sub ordering keys, Tumbling, Sliding, and Session Windows, Watermark Triggers, Allowed Lateness. |
| **Chapter 04: Stateful Processing** | `@StateId` (`ReadModifyWriteStateSpec`, `BagStateSpec`), `@TimerId` state cleanup, `withHotKeyFanout`. |
| **Chapter 05: Performance & Monitoring** | Dataflow Streaming Engine, Dataflow Shuffle, System Lag vs. Data Freshness, Cloud Monitoring SLA Alert Policies. |
| **Chapter 06: Resilient Operations** | Zero-downtime rolling updates (`--update`), transform name compatibility, Pub/Sub 24h seek replay disaster recovery. |
| **Chapter 07: Testing, CI/CD & Security** | `TestPipeline`, `PAssert`, `TestStream`, Docker Flex Templates, Cloud Build CI/CD, Private IPs, and CMEK. |

You are now fully equipped to build, test, operate, and secure enterprise-scale stream and batch data processing systems on Google Cloud Dataflow!